In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline
import scipy.io as sio
from dataclasses import dataclass
from typing import List, Tuple
import os
from dotenv import load_dotenv
load_dotenv()
import tidy3d as td
from tidy3d import web
import numpy as np
from pathlib import Path
from stl import mesh
import matplotlib.pyplot as plt
import re

In [ ]:
import sys
import os

# Assuming /AutomationModule is in the root directory of your project
sys.path.append(os.path.abspath(fr'H:\phd stuff\tidy3d'))

from AutomationModule import * 

import AutomationModule as AM

In [ ]:
tidy3dAPI = os.environ["API_TIDY3D_KEY"]


In [ ]:
lambdas = np.array([10,2.5])
sphere,phi,theta=AM.get_sphere(1200)
run = True


In [ ]:
from pdb import run


folder_path = rf"./Structures"
project_name = "20260213_LSU_get_g_factor_size_14p3"
postprocess_results = []
runtime_ps = 25e-12
min_steps_per_lambda = 20
cuts=[1]
h5_bg = None
ref = True
for direction in ["z"]: 
    for dirpath, dirnames, filenames in os.walk(folder_path):
        try:
            for filename in filenames:
                if filename.endswith(".h5"):
                    ff = float(re.search(r'ff_([+-]?\d+(?:\.\d+)?)', filename).group(1))
                    n_value = float(re.search(r'n_([+-]?\d+(?:\.\d+)?)', filename).group(1))
                    for cut in cuts:
                        if not (Path(filename).suffix==".h5" or Path(filename).suffix==".stl"):
                            continue 
                        if os.path.isfile(os.path.join(dirpath, filename)):
                            file=os.path.join(dirpath, filename)
                            structure_1 = AM.loadAndRunStructure(key = tidy3dAPI, file_path=file
                                                            ,direction=direction, lambda_range=lambdas,
                                                            box_size=14.3,runtime_ps=runtime_ps,min_steps_per_lambda=min_steps_per_lambda,
                                                           scaling=1,shuoff_condtion=1e-20, verbose=True, 
                                                           monitors=[], flux_monitor_position=18,cell_size_manual=20,
                                                           freqs=250, boundaries="absorbers",
                                                           cut_condition=cut, source="planewave", absorbers=100, use_permittivity=False,sim_name=rf"{Path(filename).stem}_size_{cut}" + (rf"_bg_{h5_bg}" if h5_bg else ""),h5_bg=h5_bg,
                                                           )
                            sim=structure_1.sim
                            monitor_field = td.FieldProjectionAngleMonitor(
                                    center=[0.0, 0.0, 0.0],
                                    
                                    size=(structure_1.t_slab_x+1, structure_1.t_slab_y+1, structure_1.t_slab_z+1),
                                    name="field_monitor",
                                    freqs =structure_1.monitor_freqs,
                                    phi=list(phi),
                                    theta=list(theta),
                                    proj_distance=1e6,  # Set a very large projection distance to approximate far-field conditions
                                    far_field_approx=True, 
                                        )
                            source = td.TFSF(
                                center=(0, 0, 0),
                                size=(structure_1.t_slab_x+0.5, structure_1.t_slab_y+0.5, structure_1.t_slab_z+0.5),
                                source_time=structure_1.gaussian_pulse,
                                injection_axis=2,  # inject along the z axis...
                                direction="+",  # ...in the positive direction, i.e. along z+
                                name="tfsf1",
                                pol_angle=0,
                            )
                            sim=sim.copy(update={"size":(structure_1.t_slab_x+4, structure_1.t_slab_y+4, structure_1.t_slab_z+4),"monitors":[monitor_field],"sources":[source]})
                            folder_desc = rf"H:\phd stuff\tidy3d\data\{project_name}\n_{n_value:.2f}"
                            sim.plot_3d()
                            fig, ax = plt.subplots(1, tight_layout=True, figsize=(16, 8))
                            sim.plot_eps(z=0, freq=structure_1.monitor_freqs[0], ax=ax)
                            if run:
                                os.makedirs(folder_desc, exist_ok=True)
                                sim_name=rf"LSU_{Path(filename).stem}_size_{cut}"
                                if os.path.exists(os.path.join(folder_desc, sim_name+".txt")):
                                    print("Exist!")
                                else:
                                    id =web.upload(sim, folder_name=project_name,task_name=sim_name, verbose=True)
                                    ids = '\n' + id
                                    with open(os.path.join(folder_desc, sim_name+".txt"), "w") as file:
                                        # Write the string to the file
                                        file.write(ids)
                                    web.start(task_id = id)
                                    web.monitor(id)
                            

        except Exception as e:
            print(f"Error processing {dirpath}: {e}")
        
    

